<a href="https://colab.research.google.com/github/lunchtrey84/MSAI_631_REPO/blob/main/MSAI_631_Design_a_Conversational_Agent_Exercise_Trey_Williams.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<h1 align="center">
<h1 align="center"> Design a Conversational Agent Exercise


<h3 align="center"> Trey Williams

<h3 align="center"> University of the Cumberlands

<h3 align="center"> MSAI-631-A01: Artificial Intelligence for Human-Computer Interaction
    
<h3 align="center"> Dr. Alan Dennis

In [8]:
from google.colab import files
uploaded = files.upload()

Saving movie_statistic_dataset.csv to movie_statistic_dataset (1).csv


In [24]:
import pandas as pd
import re

movies_df = pd.read_csv("movie_statistic_dataset.csv")

movies_df["production_date"] = pd.to_datetime(
    movies_df["production_date"]
)

movies_df["year"] = movies_df["production_date"].dt.year

In [25]:
# --------------------------------------------------
# AZURE SENTIMENT ANALYSIS
# --------------------------------------------------

def analyze_sentiment(text):
    """
    Replace this function with your existing Azure sentiment function.
    It should return:
        positive
        negative
        neutral
        mixed
    """

    # Temporary local fallback for testing
    positive_words = [
        "yes", "yeah", "yep", "great", "good",
        "love", "like", "perfect", "awesome"
    ]

    negative_words = [
        "no", "nope", "hate", "bad",
        "terrible", "don't like", "not good"
    ]

    text = text.lower()

    if any(word in text for word in positive_words):
        return "positive"

    if any(word in text for word in negative_words):
        return "negative"

    return "neutral"

In [26]:
# --------------------------------------------------
# BUILD GENRE LIST FROM DATASET
# --------------------------------------------------

all_genres = set()

for genre_list in movies_df["genres"].dropna():
    for genre in genre_list.split(","):
        all_genres.add(genre.strip())

In [27]:
# --------------------------------------------------
# NATURAL LANGUAGE GENRE MAPPINGS
# --------------------------------------------------

genre_aliases = {
    "funny": "Comedy",
    "comedic": "Comedy",

    "scary": "Horror",
    "creepy": "Horror",
    "frightening": "Horror",

    "romantic": "Romance",
    "love story": "Romance",

    "suspenseful": "Thriller",
    "suspense": "Thriller",

    "science fiction": "Sci-Fi",
    "sci fi": "Sci-Fi",
    "sci-fi": "Sci-Fi",

    "animated": "Animation",
    "cartoon": "Animation"
}

In [35]:
# --------------------------------------------------
# MOVIE RECOMMENDATION FUNCTION
# --------------------------------------------------

def recommend_movie(user_input, suggested_movies):
    results = movies_df.copy()

    text = user_input.lower()

    selected_genres = []

    # Detect descriptive words
    for phrase, genre in genre_aliases.items():
        if phrase in text:
            selected_genres.append(genre)

    # Detect actual genre names from dataset
    for genre in all_genres:
        if genre.lower() in text:
            selected_genres.append(genre)

    # Remove duplicates
    selected_genres = list(set(selected_genres))

    # Filter for every selected genre
    for genre in selected_genres:
        results = results[
            results["genres"].str.contains(
                genre,
                case=False,
                na=False
            )
        ]

    # --------------------------------------------------
    # YEAR FILTERS
    # --------------------------------------------------

    after_year = re.search(
        r"after\s+(\d{4})",
        text
    )

    if after_year:
        year = int(after_year.group(1))

        results = results[
            results["year"] > year
        ]


    before_year = re.search(
        r"before\s+(\d{4})",
        text
    )

    if before_year:
        year = int(before_year.group(1))

        results = results[
            results["year"] < year
        ]


    # Decades such as 1980s, 1990s, 2000s
    decade_match = re.search(
        r"(19\d0|20\d0)s",
        text
    )

    if decade_match:
        decade = int(decade_match.group(1))

        results = results[
            (results["year"] >= decade) &
            (results["year"] <= decade + 9)
        ]


    # --------------------------------------------------
    # RATING FILTERS
    # --------------------------------------------------

    rating_match = re.search(
        r"(?:rated|rating|above|over|at least)\s*(\d(?:\.\d)?)",
        text
    )

    if rating_match:
        minimum_rating = float(
            rating_match.group(1)
        )

        results = results[
            results["movie_averageRating"]
            >= minimum_rating
        ]


    if "highly rated" in text:
        results = results[
            results["movie_averageRating"] >= 7.5
        ]


    # --------------------------------------------------
    # RUNTIME FILTERS
    # --------------------------------------------------

    runtime_match = re.search(
        r"(?:under|less than)\s*(\d+)\s*(?:minutes|mins)",
        text
    )

    if runtime_match:
        max_runtime = int(
            runtime_match.group(1)
        )

        results = results[
            results["runtime_minutes"]
            <= max_runtime
        ]


    hours_match = re.search(
        r"(?:under|less than)\s*(\d+)\s*hours?",
        text
    )

    if hours_match:
        hours = int(
            hours_match.group(1)
        )

        results = results[
            results["runtime_minutes"]
            <= hours * 60
        ]


    # --------------------------------------------------
    # REMOVE MOVIES ALREADY SUGGESTED
    # --------------------------------------------------

    results = results[
        ~results["movie_title"].isin(
            suggested_movies
        )
    ]


    if results.empty:
        return None


    # Rank movies
    results = results.sort_values(
        by=[
            "movie_averageRating",
            "movie_numerOfVotes"
        ],
        ascending=False
    )

    return results.iloc[0]


# --------------------------------------------------
# MOVIEMATCH CONVERSATIONAL AGENT
# --------------------------------------------------

print("MovieMatch")
print("------------------------------")

suggested_movies = []

user_input = input("\nWhat kind of movie do you want? ")

while True:
    movie = recommend_movie(user_input, suggested_movies)

    if movie is None:
        print("\nMovieMatch: I couldn't find another movie matching those preferences.")
        break

    suggested_movies.append(movie["movie_title"])

    print(
        f"\nMovieMatch: I recommend {movie['movie_title']} "
        f"({int(movie['year'])}) "
        f"with a rating of {movie['movie_averageRating']}/10."
    )

    answer = input(
        "\nMovieMatch: Is this selection acceptable? "
    ).strip()

    text = answer.lower()

    # Exit commands
    if text in ["bye", "goodbye", "exit", "quit"]:
        print("\nMovieMatch: Goodbye!")
        break

    # Run sentiment analysis
    sentiment = analyze_sentiment(answer)

    print(f"\nSentiment detected: {sentiment}")

    # Positive = accept recommendation
    if sentiment == "positive":
        print("\nMovieMatch: Great! Enjoy your movie.")
        break

    # Check whether the user entered a new movie request
    movie_terms = [
        "movie", "film",
        "action", "comedy", "horror",
        "drama", "romance", "thriller",
        "adventure", "fantasy", "crime",
        "sci-fi", "science fiction",
        "funny", "scary", "romantic",
        "rated", "rating",
        "after", "before",
        "director", "directed",
        "hours", "minutes"
    ]

    new_request = any(
        term in text
        for term in movie_terms
    )

    # Negative response
    if sentiment == "negative":
        if new_request:
            print("\nMovieMatch: Okay, let's change the search.")
            user_input = answer
            suggested_movies = []
        else:
            print("\nMovieMatch: No problem. Let me suggest another.")

        continue

    # Neutral response that contains a new request
    if new_request:
        print("\nMovieMatch: Okay, let's change the search.")
        user_input = answer
        suggested_movies = []
        continue

    # Out-of-scope response
    print(
        "\nMovieMatch: I can only help with movie recommendations. "
        "You can ask for another type of movie, say yes or no, "
        "or type 'bye' to exit."
    )

MovieMatch
------------------------------

What kind of movie do you want? I would like something funny and scary

MovieMatch: I recommend Shaun of the Dead (2004) with a rating of 7.9/10.

MovieMatch: Is this selection acceptable? No

Sentiment detected: negative

MovieMatch: No problem. Let me suggest another.

MovieMatch: I recommend Evil Dead II (1987) with a rating of 7.7/10.

MovieMatch: Is this selection acceptable? No, give me a science fiction movie

Sentiment detected: negative

MovieMatch: Okay, let's change the search.

MovieMatch: I recommend Inception (2010) with a rating of 8.8/10.

MovieMatch: Is this selection acceptable? No, I want a highly rated horror movie

Sentiment detected: negative

MovieMatch: Okay, let's change the search.

MovieMatch: I recommend Alien (1979) with a rating of 8.5/10.

MovieMatch: Is this selection acceptable? Find me an action movie after 2015

Sentiment detected: neutral

MovieMatch: Okay, let's change the search.

MovieMatch: I recommend B